# Local archives

Files already on disk: one file, a list of files, a folder, or a tree with a pattern.

In [1]:
import numpy as np
import pandas as pd

from floatmatcher.floatmatcher import FloatMatcher

### Points and method

In [2]:
df = pd.DataFrame({
    "longitude": [-38.2, -35.6, -31.9, -27.4, -23.1],
    "latitude":  [31.5, 34.2, 37.8, 40.1, 43.6],
    "date": pd.to_datetime(["2018-01-02 03:10", "2018-01-03 14:45", "2018-01-05 08:20",
                            "2018-01-06 21:05", "2018-01-08 11:30"]),
})

my_fm = FloatMatcher()
my_fm.set_points(df)
my_fm.set_method(type="nearest", radius=25, time_gap=np.timedelta64(12, "h"))

root = "/runtime/data/era5_daily"  # daily ERA5 files, adapt to your archive

### One file

In [3]:
my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"],
                  path=f"{root}/2018/01/era5_single-levels_20180102.nc")
my_fm.match()
my_fm.result  # only the point of 2 January is matched


    variables: dict_keys(['sst'])                 
    mean_dist:4.74 (km)
    mean_time_delta:11400.0 (seconds)                 
    nb_match_found:1

### List of files

In [4]:
files = [f"{root}/2018/01/era5_single-levels_201801{day:02d}.nc" for day in range(1, 10)]

my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"], path=files)
my_fm.match()
my_fm.result


    variables: dict_keys(['sst'])                 
    mean_dist:10.72 (km)
    mean_time_delta:9360.0 (seconds)                 
    nb_match_found:5

### Folder

Every `.nc` file under it is opened.

In [5]:
my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"],
                  path=f"{root}/2018/01")
my_fm.match()
my_fm.product.src_dataset.sizes  # the 31 days of January

Frozen({'time': 62, 'lat': 721, 'lon': 1440})

### Tree and pattern

Only the files of the days of the points are opened, and the days around.

In [6]:
my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"],
                  path=root,
                  pattern="{year}/{month:02d}/era5_single-levels_{year}{month:02d}{day:02d}.nc",
                  )
my_fm.match()
my_fm.product.src_dataset.sizes  # 9 days instead of 31

Frozen({'time': 18, 'lat': 721, 'lon': 1440})

In [7]:
my_fm.reinject()

,longitude,latitude,date,sst_coloc
0,-38.2,31.5,2018-01-02 03:10:00,295.034180
1,-35.6,34.2,2018-01-03 14:45:00,293.568359
2,-31.9,37.8,2018-01-05 08:20:00,291.394775
3,-27.4,40.1,2018-01-06 21:05:00,290.468750
4,-23.1,43.6,2018-01-08 11:30:00,288.234375


### 2D grid: look-up table

No time axis: points without time, `time_gap` not needed.

In [8]:
lut_fm = FloatMatcher()
lut_fm.set_points(df.drop(columns="date"), time=None)
lut_fm.set_product(type="lut", source="local", selected_variables=["fluorescence_chlorophyll_ratio"],
                   path="/runtime/data/LUT/LUT_v2_202608.nc")
lut_fm.set_method(type="nearest", radius=100)  # 1° grid
lut_fm.match()
lut_fm.reinject()

,longitude,latitude,fluorescence_chlorophyll_ratio_coloc
0,-38.2,31.5,1.170904
1,-35.6,34.2,1.162179
2,-31.9,37.8,1.426927
3,-27.4,40.1,1.593109
4,-23.1,43.6,1.982528
